# NIH ChestX-ray14 original-pixel ingestion — two-source verification

This is a **one-time infrastructure step**, not tied to any single phase-1 endpoint. It
downloads the full NIH ChestX-ray14 release from two independently-operated sources,
verifies every selected image's SHA-256 agrees across both, and writes a verified
`schema_version: 2` provenance artifact that `scripts/benchmark_kad.py` can accept as
`original_nih_pixels=true`. A bare, single-source download can never do that — see
`docs/CHEST_CLASSIFIER_RESET.md` for why.

**Before you run this:**

1. Use a Colab runtime with **at least ~100GB of free local disk**. Both sources are
   downloaded and extracted before verification (~42GB each, ~85-90GB peak) — the
   default low-disk Colab tier will not fit this.
2. This step does **not** need a GPU. Use a CPU runtime so this multi-hour,
   bandwidth-bound download does not occupy GPU quota.
3. Expect this to take a long time — it is bandwidth-bound, not compute-bound.
4. Add two Colab secrets, **`KAGGLE_USERNAME`** and **`KAGGLE_KEY`**, from a free
   Kaggle account's API token (kaggle.com → account settings → "Create New Token").
   This is a one-time setup, the same pattern as the existing `GITHUB_TOKEN`/`HF_TOKEN`
   secrets used elsewhere in this repo's notebooks.
5. Output is written to Google Drive under
   `MyDrive/doctor_assistant/kad_phase1/data/nih_original_development` as a durable
   artifact. Wiring that artifact into the per-endpoint evaluation notebook
   (`chest_classifier_build_colab.ipynb`) is a separate follow-up step, not done here.

**The two sources**, both plain HTTPS and both independently verifiable (unlike NIH's
own box.com host, which is JS-rendered with no fixed, checkable static links):

- **Kaggle** — `nih-chest-xrays/data`, fetched via the official Kaggle API.
- **Hugging Face** — `alkzar90/NIH-Chest-X-ray-dataset`, pinned to commit
  `36778e3b0e4f4b4fad31d1728d6190f3eda5b543`.

An earlier version of this notebook used a BitTorrent source instead of Kaggle.
**BitTorrent traffic is against Google Colab's free-tier acceptable-use policy** and
gets the runtime disconnected regardless of how much bandwidth it actually uses — this
version replaces it entirely with a second plain-HTTPS source so nothing here touches
a restricted protocol.

A file is only accepted once its SHA-256 agrees across both. Any disagreement, or a
file missing from one source, is a hard failure naming the exact file — never a silent
majority vote.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hamza09Hamza/doctor_assistant.git"
REPO_REF = "classifier-evaluation-colab"
REPO_DIR = Path("/content/doctor_assistant")
DRIVE_ROOT = Path("/content/drive/MyDrive/doctor_assistant/kad_phase1")

print("This notebook ingests original NIH pixels only; it does not evaluate any endpoint.")

## 1. Mount Drive and load the exact branch

Identical clone helper to `chest_classifier_build_colab.ipynb`: authentication stays
out of the repository URL, and an existing non-clean Colab clone is rejected so a
rerun cannot silently mix code versions.

In [ ]:
import base64
import os
import shlex
import subprocess

from google.colab import drive, userdata

drive.mount("/content/drive")

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None

auth_header = None
if github_token:
    encoded = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    auth_header = f"AUTHORIZATION: basic {encoded}"

def run_git(label, arguments, *, authenticated=False):
    command = ["git"]
    if authenticated and auth_header:
        command += ["-c", f"http.extraHeader={auth_header}"]
    command += [str(value) for value in arguments]
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.returncode:
        detail = (completed.stderr or completed.stdout or "unknown git error").strip()
        if github_token:
            detail = detail.replace(github_token, "***")
        if auth_header:
            detail = detail.replace(auth_header, "***")
        raise RuntimeError(f"{label} failed: {detail}")
    return completed.stdout.strip()

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git clone; use a fresh runtime")
    origin = run_git("origin lookup", ["-C", REPO_DIR, "remote", "get-url", "origin"])
    if origin != REPO_URL:
        raise RuntimeError(f"Refusing to authenticate to unexpected origin {origin!r}; use a fresh runtime")
    dirty = run_git("repository status", ["-C", REPO_DIR, "status", "--porcelain"])
    if dirty:
        raise RuntimeError("The Colab clone has local changes. Use a fresh runtime rather than mixing versions.")
    run_git("branch fetch", ["-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_REF], authenticated=True)
    run_git("branch checkout", ["-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"])
else:
    run_git(
        "repository clone",
        ["clone", "--branch", REPO_REF, "--single-branch", "--depth", "1", REPO_URL, REPO_DIR],
        authenticated=True,
    )

GIT_SHA = run_git("commit lookup", ["-C", REPO_DIR, "rev-parse", "HEAD"])
os.chdir(REPO_DIR)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Loaded {REPO_REF} at {GIT_SHA}")

## 2. Install the focused runtime, including the Kaggle API client

Both sources here are plain HTTPS — no torrent client, nothing Colab's free-tier
acceptable-use policy restricts.

In [ ]:
%pip install -q "transformers==5.12.1" "numpy==2.0.2" "scipy==1.18.0" "scikit-learn==1.9.0" "pandas==2.2.2" "Pillow==12.3.0" "datasets>=2.18,<5" "huggingface_hub>=0.23" "pyarrow>=15,<24" "matplotlib>=3.8" "gdown>=5" "timm>=0.9" "monai>=1.3" "pydicom>=2.4" "nibabel>=5.2" "PyYAML>=6" "pydantic>=2.6" "kaggle>=1.6"

In [ ]:
import os
import sys

from google.colab import userdata

try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception as error:
    raise RuntimeError(
        "This notebook needs the KAGGLE_USERNAME and KAGGLE_KEY Colab secrets "
        "(kaggle.com -> account settings -> Create New Token). See the intro cell."
    ) from error

print("Python:", sys.version.split()[0])
print("Kaggle credentials loaded for user:", os.environ["KAGGLE_USERNAME"])

## 3. Build the canonical expert manifest (development cohort)

Shared across every phase-1 endpoint — this only needs to run once, regardless of
which endpoint is evaluated later.

In [ ]:
import json

def run_step(arguments):
    command = [str(value) for value in arguments]
    print("$", shlex.join(command))
    subprocess.run(command, cwd=REPO_DIR, check=True)

METADATA_DIR = Path("/content/nih_metadata_pinned")
MANIFEST_DIR = DRIVE_ROOT / "data" / "expert_manifest"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

run_step([
    sys.executable, "scripts/fetch_nih_metadata.py",
    "--output-dir", METADATA_DIR,
])

CANONICAL_MANIFEST = MANIFEST_DIR / "nih_expert_labels.csv"
CANONICAL_METADATA = MANIFEST_DIR / "nih_expert_labels.metadata.json"
run_step([
    sys.executable, "scripts/prepare_nih_expert_manifest.py",
    "--combined-csv", METADATA_DIR / "google2019_nih-chest-xray-labels.csv.gz",
    "--official-train-val-list", METADATA_DIR / "train_val_list.txt",
    "--official-test-list", METADATA_DIR / "test_list.txt",
    "--output-csv", CANONICAL_MANIFEST,
    "--output-json", CANONICAL_METADATA,
    "--output-cohort", "development",
])
print("Canonical manifest:", CANONICAL_MANIFEST)

## 4. Fetch Source A — Kaggle (`nih-chest-xrays/data`)

Downloads via the official Kaggle API (plain HTTPS, no restricted protocol), then
walks the extracted tree filtering to canonical NIH filenames only -- defensive
against whatever internal folder layout Kaggle's archive uses, rather than assuming a
fixed path.

In [ ]:
import shutil
import zipfile

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from data.nih_protocol import is_nih_image_filename

KAGGLE_DATASET = "nih-chest-xrays/data"
SOURCE_A_RAW = Path("/content/nih_source_a_raw")
SOURCE_A = Path("/content/nih_source_a")
SOURCE_A_RAW.mkdir(parents=True, exist_ok=True)
SOURCE_A.mkdir(parents=True, exist_ok=True)

run_step([
    "kaggle", "datasets", "download",
    "-d", KAGGLE_DATASET,
    "-p", SOURCE_A_RAW,
])

archives = sorted(SOURCE_A_RAW.glob("*.zip"))
if not archives:
    raise RuntimeError(f"kaggle datasets download produced no .zip file in {SOURCE_A_RAW}")

extracted = 0
for archive_path in archives:
    print("Extracting", archive_path.name)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            name = Path(member.filename).name
            if member.is_dir() or not is_nih_image_filename(name):
                continue
            with archive.open(member) as source_file:
                (SOURCE_A / name).write_bytes(source_file.read())
            extracted += 1
    archive_path.unlink()  # free disk progressively; we only need the extracted PNGs
print(f"Source A: {extracted:,} canonical NIH image file(s) extracted to {SOURCE_A}")

## 5. Fetch Source B — Hugging Face mirror (pinned commit)

Plain HTTPS, no torrent client needed. Each shard's zip is deleted immediately after
extraction to keep peak local disk down.

In [ ]:
import zipfile

HF_REPO = "alkzar90/NIH-Chest-X-ray-dataset"
HF_REVISION = "36778e3b0e4f4b4fad31d1728d6190f3eda5b543"
SOURCE_B_RAW = Path("/content/nih_source_b_raw")
SOURCE_B = Path("/content/nih_source_b")
SOURCE_B_RAW.mkdir(parents=True, exist_ok=True)
SOURCE_B.mkdir(parents=True, exist_ok=True)

extracted = 0
for shard in range(1, 13):
    shard_name = f"images_{shard:03d}.zip"
    url = f"https://huggingface.co/datasets/{HF_REPO}/resolve/{HF_REVISION}/data/images/{shard_name}"
    destination = SOURCE_B_RAW / shard_name
    print(f"Downloading {shard_name} ({shard}/12)...")
    _download(url, destination)
    with zipfile.ZipFile(destination) as archive:
        for member in archive.infolist():
            name = Path(member.filename).name
            if member.is_dir() or not is_nih_image_filename(name):
                continue
            with archive.open(member) as source_file:
                (SOURCE_B / name).write_bytes(source_file.read())
            extracted += 1
    destination.unlink()
print(f"Source B: {extracted:,} canonical NIH image file(s) extracted to {SOURCE_B}")

## 6. Verify both sources agree, then ingest

Runs `scripts/fetch_nih_original_images.py`, which hard-fails, naming the exact file,
on any disagreement or insufficient source coverage — never a silent majority vote.

In [ ]:
LOCAL_OUTPUT = Path("/content/nih_original_development")

run_step([
    sys.executable, "scripts/fetch_nih_original_images.py",
    "--manifest", CANONICAL_MANIFEST,
    "--output-images-dir", LOCAL_OUTPUT,
    "--cohort", "development",
    "--source", f"Kaggle {KAGGLE_DATASET}={SOURCE_A}",
    "--source", f"Hugging Face {HF_REPO}@{HF_REVISION[:12]}={SOURCE_B}",
])

provenance_path = LOCAL_OUTPUT / "ORIGINAL_NIH_PIXELS.provenance.json"
provenance = json.loads(provenance_path.read_text())
print("schema_version:", provenance["schema_version"])
print("original_nih_pixels:", provenance["original_nih_pixels"])
print("images verified:", len(provenance["images"]))
print(
    "minimum_independent_sources:",
    provenance["archive_verification"]["minimum_independent_sources"],
)

## 7. Persist the verified artifact to Drive

In [ ]:
import shutil

DRIVE_OUTPUT = DRIVE_ROOT / "data" / "nih_original_development"
if DRIVE_OUTPUT.exists():
    raise FileExistsError(
        f"{DRIVE_OUTPUT} already exists; remove or rename it before re-running this "
        "notebook so a partial or stale copy is never silently mixed with a new run"
    )
print(
    f"Copying {len(provenance['images']):,} verified image(s) to Drive; this can "
    "take a while over Drive FUSE..."
)
shutil.copytree(LOCAL_OUTPUT, DRIVE_OUTPUT)
print("Durable artifact saved to:", DRIVE_OUTPUT)

## Done — what this produced, and what's still open

This produced a verified `ORIGINAL_NIH_PIXELS.provenance.json` plus its images in
`MyDrive/doctor_assistant/kad_phase1/data/nih_original_development`, generated from
two independently-operated sources agreeing on every selected file's bytes. Per
`docs/CHEST_CLASSIFIER_RESET.md`, this is the artifact that can eventually unlock
Gate 4 acceptance evidence for any phase-1 endpoint.

**Not done here — a separate follow-up:**

- `chest_classifier_build_colab.ipynb`'s benchmark step does not yet point at this
  output. Wiring `--image-root`/`--image-provenance` to this Drive path for the
  development cohort is the next step, not part of this notebook.
- The locked-test cohort was never touched; this only ever fetched
  `cohort=development` images.
- Peak local disk during this run was ~85-90GB. Shrinking that further would mean
  changing `scripts/fetch_nih_original_images.py` to compare precomputed hash lists
  instead of two on-disk directories, so only one source's raw bytes are kept at a
  time. Not built yet.